# Equivalenze

In [2]:
from QuantumCompiling import CircuitState, CircuitShape
import scipy.sparse as sp
import numpy as np
import itertools as it
import csv

In [3]:
n_qubits = 4
gates = { 
    1 : [ ["ID"] ],
    2 : [ ["Sc","St"],["CZc","CZt"] ],
    3 : [ ["Sc","ID","St"] ],
    4 : [ ["Sc","Sc","St","St"] ] ,
}

shape = CircuitShape(n_qubits,gates)
shape.print_steps(); 

0: ['ID', 'ID', 'ID', 'ID']
1: ['ID', 'ID', 'Sc', 'St']
2: ['ID', 'ID', 'CZc', 'CZt']
3: ['ID', 'Sc', 'St', 'ID']
4: ['ID', 'CZc', 'CZt', 'ID']
5: ['ID', 'Sc', 'ID', 'St']
6: ['Sc', 'St', 'ID', 'ID']
7: ['Sc', 'St', 'Sc', 'St']
8: ['Sc', 'St', 'CZc', 'CZt']
9: ['CZc', 'CZt', 'ID', 'ID']
10: ['CZc', 'CZt', 'Sc', 'St']
11: ['CZc', 'CZt', 'CZc', 'CZt']
12: ['Sc', 'ID', 'St', 'ID']
13: ['Sc', 'Sc', 'St', 'St']



d = 1, nessun time-step è equivalente a sè stesso

In [4]:
available_steps = shape.get_steps()
for i in range(len(available_steps)):
    c1 = CircuitState(shape,[available_steps[i]])
    for j in range(i+1,len(available_steps)):
        c2 = CircuitState(shape,[available_steps[j]])
        if(CircuitState.equivalent(c1,c2)):
            print(c1.draw())
            print(c2.draw())

d=2, si costruiscono tutti i circuiti a due step (effettivi, la classe CircuitState associa lo stesso stato a quelli banalmente equivalenti e se possibile semplifica ad un numero minore di timesteps; va dunque trattato un solo circuito per classe di banale equivalenza e quelli contraibili vanno scartati) e si confrontano tra loro

In [5]:
available_steps = shape.get_steps()

circuits = []
ids = []

for i in range(len(available_steps)):
    for j in range(len(available_steps)):
        c = CircuitState(shape, [available_steps[i],available_steps[j]])
        if c.get_n_steps() == 2 and c not in ids: 
            if not c.isID():
                if all(not c.equals(c1) for c1 in circuits):
                    circuits.append(c)
            else:
                ids.append(c)
                
print("Circuiti effettivi:", len(circuits))
# for c in circuits: print(c.draw())
# print()
# print()
print("Identità:", len(ids))
# for c in ids: print(c.draw())

Circuiti effettivi: 129
Identità: 13


In [6]:
n_eq = 0
n_confronti = 0
for i in range(len(circuits)-1):
    for j in range(i+1, len(circuits)):
        n_confronti += 1
        c1 = circuits[i]
        c2 = circuits[j]
        if CircuitState.equivalent(c1,c2): n_eq +=1
        # print("Contfronto tra:",CircuitState.equivalent(c1,c2))
        # print(c1.draw())
        # print(c2.draw())
        # print()
print("Numero di confronti:",n_confronti)
print("Numero di equivalenze:",n_eq)

Numero di confronti: 8256
Numero di equivalenze: 77


In generale per d timesteps

In [7]:
d=2

actual_circuits = []
IDequivalent = []
available_steps = shape.get_steps()

ids = list(it.product(range(len(available_steps)), repeat=d))
for i in range(len(ids)):
    c = CircuitState(shape, [available_steps[k] for k in ids[i]])
    if c.get_n_steps() == d and c not in IDequivalent:
        if c.isID():
            IDequivalent.append(c)
        elif all(not c.equals(c1) for c1 in actual_circuits):
            actual_circuits.append(c)
print("Circuiti effettivi trovati:",len(actual_circuits))
print("Circuiti identità:",len(IDequivalent))

Circuiti effettivi trovati: 129
Circuiti identità: 13


act = [10,66,568,4050]
id = [1,10,0,350]

Per trovare le equivalenze ora basta semplicemente confrontare ogni circuito, escludendo l'ultimo, con tutti i successivi della lista

In [8]:
n_eq = 0
for i in range(len(actual_circuits)-1):
    for j in range(i+1,len(actual_circuits)):
        if CircuitState.equivalent(actual_circuits[i],actual_circuits[j]):
            n_eq += 1
print("Numero di equivalenze:",n_eq)

Numero di equivalenze: 77


Le equivalenze vanno salvate su file, per farlo si usano gli indici i e j. In questo modo il file dipende strettamente dalla shape utilizzata, per questo anch'essa deve stare nel file

In [ ]:
import csv

filename = "csv/eq2_c.csv"
with open(filename, 'w', newline='') as fout:
    writer = csv.writer(fout, delimiter="\t")
    writer.writerow(["n_qubits:" +str(n_qubits)])
    writer.writerow(["gates:"])
    for key in gates.keys():
        writer.writerow(gates[key])
    writer.writerow(["equivalences:"])
    for i in range(len(actual_circuits)-1):
        print("C1:",str(i+1)+"/"+str(len(actual_circuits)-1))
        for j in range(i+1,len(actual_circuits)):
            if CircuitState.equivalent(actual_circuits[i],actual_circuits[j]):
                writer.writerow( [shape.get_step_index(step) for step in actual_circuits[i].get_steps()]
                                + [shape.get_step_index(step) for step in actual_circuits[j].get_steps()])

Si sposta il codice nel metodo find_equivalences della classe CircuitShape

In [ ]:
n_qubits = 4
gates = { 
    1 : [ ["ID"] ],
    2 : [ ["Sc","St"],["CZc","CZt"] ]
    # ,3 : [ ["Sc","ID","St"] ],
    # 4 : [ ["Sc","Sc","St","St"] ] ,
}

CircuitShape.find_equivalences(gates,n_qubits,4,"prova")

Testistiamo ora le equivalenze estraendole dai file e costruendo e confrontando i circuiti indicati

In [10]:
n_qubits = 4
gates = { 
    1 : [ ["ID"] ],
    2 : [ ["Sc","St"],["CZc","CZt"] ] 
}
shape = CircuitShape(n_qubits,gates)


d = 3
filename = f"equivalenze/eq_d={d}.csv"

with open(filename, 'r', newline='') as fin:
    reader = csv.reader(fin, delimiter="\t")
    #skips the header
    row = None
    while(row != "equivalences:"):
        row = next(reader)[0]
    
    equivalences = [[int(id) for id in row] for row in reader]

i = 6
for eq in equivalences:
    c1 = CircuitState.from_steps_ids(eq[:d],shape)
    c2 = CircuitState.from_steps_ids(eq[d:],shape)
    
    if CircuitState.equivalent(c1,c2):
        print(i,":","Equivalenza!")
        # print(c1.draw())
        # print(c2.draw())
        # print()
    else:
        raise("Non-equivalenza trovata")
    i = i+1


6 : Equivalenza!
7 : Equivalenza!
8 : Equivalenza!
9 : Equivalenza!
10 : Equivalenza!
11 : Equivalenza!
12 : Equivalenza!
13 : Equivalenza!
14 : Equivalenza!
15 : Equivalenza!
16 : Equivalenza!
17 : Equivalenza!
18 : Equivalenza!
19 : Equivalenza!
20 : Equivalenza!
21 : Equivalenza!
22 : Equivalenza!
23 : Equivalenza!
24 : Equivalenza!
25 : Equivalenza!
26 : Equivalenza!
27 : Equivalenza!
28 : Equivalenza!
29 : Equivalenza!
30 : Equivalenza!
31 : Equivalenza!
32 : Equivalenza!
33 : Equivalenza!
34 : Equivalenza!
35 : Equivalenza!
36 : Equivalenza!
37 : Equivalenza!
38 : Equivalenza!
39 : Equivalenza!
40 : Equivalenza!
41 : Equivalenza!
42 : Equivalenza!
43 : Equivalenza!
44 : Equivalenza!
45 : Equivalenza!
46 : Equivalenza!
47 : Equivalenza!
48 : Equivalenza!
49 : Equivalenza!
50 : Equivalenza!
51 : Equivalenza!
52 : Equivalenza!
53 : Equivalenza!
54 : Equivalenza!
55 : Equivalenza!
56 : Equivalenza!
57 : Equivalenza!
58 : Equivalenza!
59 : Equivalenza!
60 : Equivalenza!
61 : Equivalen

E poi le equivalenze con l'identità

In [11]:

d = 4
filename = f"equivalenze/id_d={d}.csv"

with open(filename, 'r', newline='') as fin:
    reader = csv.reader(fin, delimiter="\t")
    #skips the header
    row = None
    while(row != "id equivalences:"):
        row = next(reader)[0]
    
    id_equivalences = [[int(id) for id in row] for row in reader]

i = 6
for id_eq in id_equivalences:
    c = CircuitState.from_steps_ids(id_eq,shape)
    
    if c.isID():
        print(i,":","Identità!")
        # print(c1.draw())
        # print(c2.draw())
        # print()
    else:
        raise("Non-identità trovata")
    i = i+1


6 : Identità!
7 : Identità!
8 : Identità!
9 : Identità!
10 : Identità!
11 : Identità!
12 : Identità!
13 : Identità!
14 : Identità!
15 : Identità!
16 : Identità!
17 : Identità!
18 : Identità!
19 : Identità!
20 : Identità!
21 : Identità!
22 : Identità!
23 : Identità!
24 : Identità!
25 : Identità!
26 : Identità!
27 : Identità!
28 : Identità!
29 : Identità!
30 : Identità!
31 : Identità!
32 : Identità!
33 : Identità!
34 : Identità!
35 : Identità!
36 : Identità!
37 : Identità!
38 : Identità!
39 : Identità!
40 : Identità!
41 : Identità!
42 : Identità!
43 : Identità!
44 : Identità!
45 : Identità!
46 : Identità!
47 : Identità!
48 : Identità!
49 : Identità!
50 : Identità!
51 : Identità!
52 : Identità!
53 : Identità!
54 : Identità!
55 : Identità!
56 : Identità!
57 : Identità!
58 : Identità!
59 : Identità!
60 : Identità!
61 : Identità!
62 : Identità!
63 : Identità!
64 : Identità!
65 : Identità!
66 : Identità!
67 : Identità!
68 : Identità!
69 : Identità!
70 : Identità!
71 : Identità!
72 : Identità!